# PyTorch 权重持久化完整详解
## 全文学习主线逻辑
第一部分：底层基础原理（搞懂底层规则、风险、存储机制，为后续交付方案做理论支撑）
1. 序列化基础与 torch.save / torch.load 通用工具本质
2. pickle底层机制与安全漏洞根源
3. nn.Module 核心：记录网络结构、权重的内置属性与配套方法 + state_dict有序权重逻辑
4. pth文件两种存储结构：静态权重映射 / 完整可执行模型
5. 工程兼容场景：跨设备、多卡、迁移学习
6. 通用工程工具封装
7. 安全拓展：safetensors 无pickle权重格式

第二部分：落地交付三大方案（全文应用主线，基于前文原理做选型落地）
1. 方案A：单文件state_dict权重 weights.pth（纯原生，无配套结构文件）
2. 方案B：原生双文件 weights.pth + model_arch.py（权重+配套网络源码，团队内部）
3. 方案C：Hugging Face一体化包 config.json + model.safetensors（开源商用交付行业标准）

### 完整目录
1. 底层基础：序列化、torch.save通用工具、pickle风险
2. 权重有序存储核心：nn.Module 记录结构/权重的属性、方法与 state_dict
3. pth文件底层拆解：两种pickle存储说明书
4. 四大存储落地场景：权重文件/完整模型/训练断点/TorchScript部署
5. 工程兼容适配：跨设备、多卡、迁移学习
6. 高频报错避坑指南
7. 通用工程工具函数封装
8. 拓展：safetensors 无pickle安全权重格式
9. 核心应用主线：对外交付三大方案选型对比

# 一、底层基础：序列化、torch.save / torch.load 与 pickle 机制
## 1. 什么是序列化
序列化：将内存中复杂、带嵌套引用的程序对象，整理成**线性有序二进制字节序列**写入磁盘永久保存；
反序列化：读取磁盘二进制字节，还原为内存可运行的程序对象。
核心关键词：序列 = 规整有序线性数据流，二进制仅为存储载体，不是序列化的核心定义。

## 2. torch.save 只是通用序列化工具
通用含义：该函数不专门用于保存神经网络模型，可序列化任意Python对象：张量、字典、数字、自定义类、优化器等；
决定文件存储内容的不是torch.save，而是传入的对象：
- 传入model.state_dict()：仅序列化有序权重映射；
- 传入完整model实例：序列化整套可执行网络重建代码,包含：网络结构定义 + 所有权重 / 缓冲区

### 底层实现依赖 pickle
`torch.save` 能够实现任意Python对象通用序列化，底层全部依托Python标准库`pickle`完成对象拆解、层级记录、二进制编码；
但pickle本身的序列化设计存在原生安全缺陷，这也是PyTorch 2.6版本新增`weights_only`安全拦截机制的根源。

#### `weights_only=True` 底层拦截逻辑

加载 pickle 文件时，PyTorch 会限制 pickle 能还原的对象白名单，**只允许以下安全类型**：

- 基础数值：int/float/bool
- 基础容器：list/tuple/dict
- PyTorch 安全张量 `torch.Tensor`
- 简单字符串、字节

直接拦截所有**自定义 Python 类实例**的反序列化：

- 遇到自定义网络类、自定义对象、带 `__reduce__` 的恶意类 → 直接抛出 `UnpicklingError` 拦截；
- 只放行纯权重字典 `state_dict`、纯张量集合，彻底断绝代码执行入口。

## 3. pickle底层原生安全风险
pickle序列化会记录对象创建、类导入、实例化完整执行指令；反序列化时逐条执行，加载来历不明的`.pth`文件会触发远程代码执行高危漏洞；
PyTorch2.6+ 默认开启 weights_only=True，自动拦截包含可执行重建代码的完整model文件。

## 两种基础存储范式前置对比
| 存储方式 | 核心存储内容 | 优缺点 | 适用范围 |
|------|----------|--------|----------|
| state_dict 权重字典 | 仅参数名-张量静态映射，无执行代码 | 体积小、跨版本兼容、安全 | 推理、分发、训练断点（生产推荐） |
| 完整model对象 | 序列化整个nn.Module，附带网络重建执行指令 | 本地无需手写网络，但安全差、兼容性极差 | 仅本地单人临时调试，生产环境禁用 |

# 二、权重有序存储核心：nn.Module 体系与 state_dict
## 承接上文：两种存储方案差异的根源
上一节我们知道torch.save分两种传参，文件内容天差地别：
1. torch.save(完整model)：同时保存网络结构 + 所有权重，带可执行重建代码，有安全漏洞
2. torch.save(model.state_dict())：只保存权重数值映射，无网络结构、无执行代码，生产安全
差异根源：`nn.Module` 内置一整套**专门记录网络结构、权重的属性与配套方法**，下面完整梳理。

## 2.1 nn.Module 记录【网络结构】的内置属性
### 1）self._modules（最核心结构属性，有序字典）
自动收集类 __init__ 中以 self.xxx 定义的所有子层（Conv/BN/Linear/自定义子Module），严格保留定义先后顺序，完整记录网络层级、嵌套结构；
序列化完整model时，pickle会把 _modules 整体打包存入文件，因此能重建网络骨架。
作用区分：_modules 只记录「当前模块挂载了哪些子层、子网络」，只存层实例对象，**不存储任何权重数值**。

### 2）self._parameters（有序字典，单层参数容器）
存储当前模块自身的可训练参数 weight/bias；
关键说明：只存本层内部权重张量，**不包含下层子模块的参数**。
含义拆解：
1. 当前模块：单独一个独立nn.Module（顶层网络、Conv、Linear、BN、自定义子块都算独立模块）；
2. 下层子模块：当前模块通过self.xxx挂载在内部的其他完整层/子网络；
3. 边界规则：每个模块的_parameters只保管自己的weight/bias，下层子模块的参数只存在子模块自身的_parameters里，不会自动向上汇总到上层。

### 3）self._buffers（有序字典，单层缓冲区容器）
存储当前模块不参与梯度更新的持久数值：BN running_mean、running_var 等；
和_parameters规则一致：仅存储当前模块自身缓冲区，不递归抓取下层子模块缓冲区。

### 4）self._modules、_parameters、_buffers 关系
- _modules：存**子网络结构、层骨架**（只记录有哪些层，无数值）；
- _parameters + _buffers：存**当前层专属权重与统计数值**，互不跨层共享；
递归遍历全部子模块，才能集齐完整网络结构+全部权重。

## 2.2 举嵌套网络实例直观理解三者边界


In [1]:
# 导入PyTorch核心库、神经网络模块
import torch
import torch.nn as nn

# 定义子块嵌套模块（复合子网络，内部包含单层Linear算子）
class SubBlock(nn.Module):
    def __init__(self):
        # 父类nn.Module初始化，必须执行，否则无法生成_modules/_parameters/_buffers
        super().__init__()
        # 挂载单层线性层，存入当前SubBlock实例的_modules有序字典
        self.linear = nn.Linear(10, 5)

# 顶层主网络（包含子块、独立线性层、顶层自定义参数）
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # 一级直属子模块：自定义复合子块SubBlock实例，存入self._modules
        self.block = SubBlock()
        # 一级直属子模块：独立单层Linear算子，存入self._modules
        self.out = nn.Linear(5, 2)
        
        # 手动注册顶层专属可训练参数：直接存入顶层Net的_parameters，不归属任何子模块
        self.register_parameter("top_bias", nn.Parameter(torch.randn(2)))

# 实例化完整主网络
model = Net()

print("===== 1. 顶层 Net 实例 =====")
# _modules：仅存储当前模块下一级直属子层/子网络，无数值权重，只记录结构骨架
print("model._modules 键（存储一级直属子模块）:", list(model._modules.keys()))
# _parameters：仅存储当前模块自身手动注册参数，子模块内部weight/bias不会上浮
print("model._parameters 键（仅存储顶层自身注册的参数，不含任何子模块参数）:", list(model._parameters.keys()), "\n")

print("===== 2. 顶层子模块 block（SubBlock实例） =====")
# 取出顶层挂载的子块实例
block = model.block
# SubBlock内部仅挂载linear层，因此_modules仅包含linear
print("block._modules 键:", list(block._modules.keys()))
# SubBlock未手动注册参数，仅内置linear子层，因此自身_parameters为空
print("block._parameters 键（block自身无注册参数，为空）:", list(block._parameters.keys()), "\n")

print("===== 3. block内部 linear 层（Linear实例） =====")
# 取出子块内部的单层线性算子
linear = block.linear
# 基础算子Linear无内部子层，_modules为空
print("linear._modules 键:", list(linear._modules.keys()))
# Linear内置weight、bias两个可训练参数，存储在自身_parameters中，隔离于父模块
print("linear._parameters 键（linear层自己的weight、bias）:", list(linear._parameters.keys()), "\n")

print("===== 4. 顶层子模块 out（Linear实例） =====")
# 取出顶层独立线性层
out = model.out
# 单层算子无内部子模块
print("out._modules 键:", list(out._modules.keys()))
# out层自有参数完全隔离，不会出现在顶层model._parameters
print("out._parameters 键（out层自己的weight、bias，不会跑到顶层model._parameters）:", list(out._parameters.keys()))

===== 1. 顶层 Net 实例 =====
model._modules 键（存储一级直属子模块）: ['block', 'out']
model._parameters 键（仅存储顶层自身注册的参数，不含任何子模块参数）: ['top_bias'] 

===== 2. 顶层子模块 block（SubBlock实例） =====
block._modules 键: ['linear']
block._parameters 键（block自身无注册参数，为空）: [] 

===== 3. block内部 linear 层（Linear实例） =====
linear._modules 键: []
linear._parameters 键（linear层自己的weight、bias）: ['weight', 'bias'] 

===== 4. 顶层子模块 out（Linear实例） =====
out._modules 键: []
out._parameters 键（out层自己的weight、bias，不会跑到顶层model._parameters）: ['weight', 'bias']


## 2.6 state_dict 只提取数值，不记录建网代码
state_dict只会递归遍历全部_modules，逐层提取两类纯数值数据：
1. 可训练参数：各层weight、bias（来自各层 _parameters）
2. 缓冲区buffer：BN均值、方差等不参与梯度更新的持久数值（来自各层 _buffers）

它**不会保存任何创建网络层的执行代码、不会序列化 _modules 结构本身**，输出结果只是一层名-张量的静态字典，这也是它没有pickle安全风险的根本原因。


In [16]:
import torch
import torch.nn as nn

# 完整建网代码（forward 必须定义，否则无法推理）
class TestNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(5, 2)
    # 前向传播逻辑，属于建网代码一部分
    def forward(self, x):
        return self.fc(x)

model = TestNet()
sd = model.state_dict()

# 1. 打印state_dict：只有参数张量，无网络类、无forward逻辑
print("state_dict 内仅参数名：", list(sd.keys()))
print("state_dict 存储内容类型：", type(sd["fc.weight"]))

# 2. 仅保存权重字典，文件不含任何网络构建/前向代码
torch.save(sd, "weights_only.pth")

# 3. 单独加载权重字典，只是纯数值映射，不能推理（无网络结构代码）
loaded_sd = torch.load("weights_only.pth", map_location="cpu", weights_only=True)
try:
    loaded_sd(torch.rand(1, 5))
except Exception as e:
    print(f"\n直接调用state_dict报错：{type(e).__name__}: {e}")
    print("原因：state_dict只是普通字典，没有网络层与forward执行逻辑")

# 4. 必须重新手写完整建网代码（__init__+forward）创建空模型，才能加载权重使用
new_model = TestNet()
new_model.load_state_dict(loaded_sd)
res = new_model(torch.rand(1, 5))
print(f"\n重建完整网络+回填权重推理结果：{res}")

state_dict 内仅参数名： ['fc.weight', 'fc.bias']
state_dict 存储内容类型： <class 'torch.Tensor'>

直接调用state_dict报错：TypeError: 'collections.OrderedDict' object is not callable
原因：state_dict只是普通字典，没有网络层与forward执行逻辑

重建完整网络+回填权重推理结果：tensor([[-0.3472, -0.3213]], grad_fn=<AddmmBackward0>)


## 2.7 nn.Module 结构/权重全套属性、方法汇总表
### 一、内置属性（存储结构/权重）
| 属性 | 存储内容 | 边界规则 |
|------|------|--------------------|
| self._modules | 有序字典，所有子层/子Module实例 | 记录网络骨架，无数值；不存储权重 |
| self._parameters | 有序字典，当前层可训练weight/bias | 仅存本层参数，不包含下层子模块参数 |
| self._buffers | 有序字典，当前层统计值张量 | 仅存本层缓冲区，不包含下层子模块缓冲区 |

### 二、配套操作方法（读取/注册/加载权重结构）
| 方法 | 功能 | 是否仅nn.Module可用 |
|------|------|--------------------|
| module.named_modules() | 递归遍历_modules，遍历全网络结构 | 是 |
| module.named_parameters() | 递归遍历所有权重参数 | 是 |
| module.named_buffers() | 递归遍历所有缓冲区 | 是 |
| module.state_dict() | 提取有序权重+缓冲区字典 | 是 |
| module.load_state_dict() | 按参数名有序回填权重 | 是 |
| module.register_parameter() | 手动注册可训练参数进 _parameters | 是 |
| module.register_buffer() | 手动注册缓冲区进 _buffers | 是 |

In [17]:
# 导入PyTorch核心运算库与神经网络层模块
import torch
import torch.nn as nn

# 自定义CNN网络类，继承nn.Module（所有网络、算子层的基类）
# 代码目标：直观验证 nn.Module._modules 有序存储规则、state_dict层级命名逻辑、区分张量与网络模块
class Net(nn.Module):
    def __init__(self):
        # 调用父类nn.Module构造函数，必须执行
        # 作用：自动初始化 _modules / _parameters / _buffers 三个有序字典容器
        super().__init__()
        # 定义第一层2D卷积：输入3通道RGB，输出16通道，卷积核3×3
        # 赋值给self.conv1，会自动存入 self._modules 有序字典，记录层级结构
        self.conv1 = nn.Conv2d(3, 16, 3)
        # 定义BN归一化层，匹配卷积输出通道16
        # 存入self._modules，顺序紧跟conv1，有序字典严格保留书写先后
        self.bn1 = nn.BatchNorm2d(16)
        # 定义全连接层：输入特征总数16*28*28，输出10分类
        # 第三个存入_modules，顺序固定
        self.fc1 = nn.Linear(16*28*28, 10)

    # 前向传播函数，定义数据流经网络的计算逻辑
    def forward(self, x):
        # 输入图像先经过卷积提取特征
        x = self.conv1(x)
        # 卷积输出送入BN做归一化（BN自带running_mean/running_var缓冲区）
        x = self.bn1(x)
        # flatten展平：从第1维开始展平，保留batch维度，转为二维向量适配全连接
        x = torch.flatten(x, 1)
        # 全连接输出分类logits
        return self.fc1(x)

# 实例化完整CNN网络对象
model = Net()

In [18]:
# model._modules是有序字典，键就是self.xxx定义的层名，顺序和__init__定义完全一致
# 打印所有直属子模块名称，验证层级存储有序性
print("_modules有序层定义顺序：", list(model._modules.keys()))

_modules有序层定义顺序： ['conv1', 'bn1', 'fc1']


In [19]:
# conv1是独立nn.Module（Conv2d算子），_parameters仅存储自身可训练参数 weight、bias
# 不会包含其他层参数，参数按层隔离
print("\n【conv1 层 _parameters】:", list(model.conv1._parameters.keys()))


【conv1 层 _parameters】: ['weight', 'bias']


In [ ]:
# BN层无可训练bias，均值、方差不参与梯度更新，统一存在 _buffers 缓冲区容器
# _buffers同样是每层独立隔离，不跨层共享
print("【bn1 层 _buffers】:", list(model.bn1._buffers.keys()))

【bn1 层 _buffers】: ['running_mean', 'running_var', 'num_batches_tracked']


In [22]:
# 提取整个网络完整权重+缓冲区有序字典
# state_dict内部逻辑：递归遍历全部_modules，逐层收集每层_parameters + _buffers
# 键命名规则：父层名.子层名.参数名，形成唯一层级标识
sd = model.state_dict()
print("\nstate_dict有序权重名称列表：")
# 遍历打印所有参数完整层级名称，直观看到命名层级关系
for name in sd.keys():
    print(name)


state_dict有序权重名称列表：
conv1.weight
conv1.bias
bn1.weight
bn1.bias
bn1.running_mean
bn1.running_var
bn1.num_batches_tracked
fc1.weight
fc1.bias


In [23]:
# -------------------------- 区分 nn.Module 和普通张量 --------------------------
# 单独创建一个线性算子（本质是nn.Module子类实例）
linear_layer = nn.Linear(10, 2)
# isinstance 判断：所有网络层、自定义网络都属于nn.Module
print("\n线性层是nn.Module实例:", isinstance(linear_layer, nn.Module))
# 算子实例可调用state_dict，输出该层自身权重字典
print("线性层权重键:", list(linear_layer.state_dict().keys()))

try:
    # 创建普通可导张量，仅数值载体，不属于任何网络模块
    test_tensor = torch.rand(10,2, requires_grad=True)
    # 普通张量没有state_dict()方法，会抛出属性异常
    test_tensor.state_dict()
except AttributeError as e:
    print("纯张量调用state_dict报错（证明仅Module支持）:", e)



线性层是nn.Module实例: True
线性层权重键: ['weight', 'bias']
纯张量调用state_dict报错（证明仅Module支持）: 'Tensor' object has no attribute 'state_dict'


In [24]:
# -------------------------- 普通张量字典 vs 网络state_dict对比 --------------------------
# 手动构造一个普通字典，仅简单存储张量，无层级、无网络结构关联
tensor_dict = {"w": torch.rand(10,2), "b": torch.rand(2)}
# 序列化保存普通张量字典
torch.save(tensor_dict, "raw_tensor_compare.pth")
print("普通张量字典保存完成，无网络有序层级管理")
# 缺陷：没有自动层级命名、不会递归收集多层参数，无法配合load_state_dict回填网络

普通张量字典保存完成，无网络有序层级管理


# 三、pth文件底层拆解：ZIP压缩包 & 两种pickle说明书
## 3.1 PyTorch1.6+ pth底层存储结构
.pth/.pt 文件本质是ZIP压缩包，内部分为两部分：
1. data/文件夹：纯二进制张量裸数据，无名称、无逻辑；
2. data.pkl：pickle序列化说明书，记录张量名称与文件编号映射/网络重建指令；
⚠️ 关键限制：完整model模式会序列化model._modules 网络结构；state_dict模式仅存权重映射，不存_modules。

## 3.2 两种pickle说明书
### 类型A：state_dict静态映射说明书（生产推荐）
仅存储「参数名称 → data下二进制文件编号」静态映射，无任何可执行网络构建代码，weights_only=True加载安全，无代码漏洞。
加载约束：使用者必须手动创建层顺序完全一致的nn.Module，依靠参数名匹配回填权重。

### 类型B：完整model可执行说明书（生产环境禁用，仅本地单人临时调试不推荐）
说明书内包含完整可执行Python指令：导入网络类、实例化、逐层读取_modules创建层、赋值权重，加载时pickle逐条执行代码，存在远程代码执行高危漏洞；
加载约束：运行环境必须存在网络类对应py文件，才能重建_modules结构，路径、版本不匹配直接崩溃；
工程红线：线上、对外交付、开源分发场景严格禁用，仅个人本地调试可临时使用。

## 3.3 底层逻辑串联总结
网络__init__有序层定义 → _modules有序字典记录完整网络结构 → state_dict递归提取各层_parameters/_buffers权重参数名 → pickle说明书有序映射 → pth压缩包存储权重。

# 四、四大存储落地场景
## 4.1 标准场景：仅保存state_dict权重（生产首选）
执行流程：定义有序nn.Module（生成完整_modules结构） → 提取state_dict有序权重映射 → torch.save保存静态映射说明书
加载流程：手动复刻同顺序网络（生成一致_modules） → torch.load读取权重映射 → load_state_dict按名回填

### state_dict关键参数strict说明
- `strict=True`（默认）：参数名、数量必须完全匹配，名称缺失/多余直接抛出报错；
- `strict=False`：自动忽略不匹配参数，用于迁移学习、网络层增删（新旧网络_modules结构不一致）场景。

In [4]:
# 导入PyTorch基础运算库与神经网络模块
import torch
import torch.nn as nn

# 定义极简基础测试网络，继承所有网络的基类nn.Module
class SimpleNet(nn.Module):
    def __init__(self):
        # 必须调用父类nn.Module的构造函数
        # 底层自动创建 _modules、_parameters、_buffers 三个有序字典容器
        super().__init__()
        # 定义单层全连接层：输入维度10，输出维度2
        # 赋值self.linear后，该层会自动存入 self._modules 有序字典记录网络结构
        self.linear = nn.Linear(10, 2)

    # 前向传播函数，规定数据计算流程，推理/训练必须依赖forward
    def forward(self, x):
        # 输入x直接经过线性层计算并返回结果
        return self.linear(x)

# 实例化网络，在内存中生成完整网络对象（包含层结构、随机初始化权重）
model = SimpleNet()

# 调用nn.Module内置state_dict()方法
# 底层逻辑：递归遍历全部子模块，收集每层_parameters可训练参数、_buffers缓冲区数值
# 返回纯静态字典：key=层级参数名，value=权重张量，不含任何网络结构、执行代码
weight_map = model.state_dict()

# torch.save：基于pickle序列化对象写入磁盘生成pth文件
# 此处仅传入state_dict权重字典，只会保存纯参数映射，无网络重建代码
# 生产环境标准写法，安全、体积小、跨版本兼容性好
torch.save(weight_map, "std_weights.pth")

# 日志打印，提示文件写入完成
print("标准state_dict权重文件保存完成")

标准state_dict权重文件保存完成


In [5]:
# 加载state_dict权重
load_model = SimpleNet()
loaded_weights = torch.load("std_weights.pth", map_location="cpu", weights_only=True)
load_model.load_state_dict(loaded_weights, strict=True)

# 推理验证
load_model.eval()
test_input = torch.randn(1, 10)
pred = load_model(test_input)
print("推理输出结果：", pred)

推理输出结果： tensor([[0.4867, 0.4223]], grad_fn=<AddmmBackward0>)


## 4.2 临时场景：保存完整model（仅本地调试，生产禁用）
底层对应类型B可执行pickle说明书，会序列化模型_modules结构与全部权重，存在安全漏洞、版本兼容差两大核心缺陷，仅用于本地快速调试，禁止对外分发、线上部署。

In [6]:
# 保存完整模型（不推荐生产使用）
torch.save(model, "full_model_temp.pt")
# PyTorch2.6+ 必须手动关闭weights_only才能加载完整model
temp_model = torch.load("full_model_temp.pt", map_location="cpu", weights_only=False)
print("完整临时模型加载完成（生产环境禁用）")

完整临时模型加载完成（生产环境禁用）


## 4.3 训练断点快照（支持完整续训）
单纯权重文件无法恢复训练优化器、学习率调度状态，因此断点是自定义字典，整合四类有序序列化对象：
1. model.state_dict() 有序权重；
2. optimizer.state_dict() 优化器梯度、动量有序状态；
3. scheduler.state_dict() 学习率调度有序参数；
4. 训练元信息：epoch、最优loss、当前学习率。

In [7]:
# 导入PyTorch优化器子模块，包含Adam、SGD等优化器与学习率调度器
import torch.optim as optim

# 自动判断运行设备：存在可用GPU则使用cuda，无GPU自动降级CPU
# torch.device生成设备标识对象，统一管理张量/模型设备迁移
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 实例化网络并将整个模型权重、缓冲区全部迁移至指定device(GPU/CPU)
train_model = SimpleNet().to(device)

# 1. 实例Adam优化器
# train_model.parameters()：获取网络所有可训练参数(weight/bias)，绑定梯度更新
# lr=1e-3：全局初始学习率 0.001
opt = optim.Adam(train_model.parameters(), lr=1e-3)

# 2. 阶梯式学习率衰减调度器StepLR
# 绑定优化器opt，每迭代step_size=10个epoch，学习率乘以gamma=0.1衰减
# 例：0~9轮 lr=0.001；10~19轮 lr=0.0001；20~29轮 lr=0.00001，以此类推
lr_sched = optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.1)

# 记录当前已经训练到第20轮，续训时从 epoch+1 开始跑
current_epoch = 20
# 记录训练过程中得到的最优损失值，用于早停、保存最优权重
best_train_loss = 0.12

# 自定义字典checkpoint：完整打包恢复训练需要的全部状态信息
# 仅保存model.state_dict无法恢复训练（丢失优化器动量、学习率调度进度）
checkpoint = {
    # 训练轮次标记，断点恢复时读取，接续下一轮训练
    "epoch": current_epoch,
    # 网络权重、BN均值方差等全部参数与缓冲区，推理/微调必需
    "model_state_dict": train_model.state_dict(),
    # 优化器内部状态：动量、梯度累积、自适应学习率统计量，无此无法无缝续训
    "optimizer_state_dict": opt.state_dict(),
    # 学习率调度器内部计数、衰减步数记录，保证续训时学习率衰减节奏不中断
    "scheduler_state_dict": lr_sched.state_dict(),
    # 历史最优损失，用于对比新epoch损失、判断是否更新最优模型
    "best_loss": best_train_loss,
    # 当前实时学习率，用于日志打印、调试观察学习率变化
    "current_lr": opt.param_groups[0]["lr"]
}

# 将完整断点字典序列化保存为pth文件
# 内部同时包含网络权重、优化器、调度器、训练元数据，支持断点无缝续训
torch.save(checkpoint, "train_checkpoint.pth")

# 控制台输出日志，提示断点文件写入完成
print("完整训练续训断点保存完成")

完整训练续训断点保存完成


In [8]:
# ====================== 1. 重建训练全套组件（必须和保存断点时超参完全一致） ======================
# 重新实例化和训练时结构完全相同的网络，并迁移到对应设备(GPU/CPU)
resume_model = SimpleNet().to(device)

# 重建Adam优化器：学习率lr、网络参数绑定关系必须和保存断点时一模一样
# 此处lr仅为初始化占位，后续会用断点里的优化器状态覆盖，不会影响真实学习率
resume_opt = optim.Adam(resume_model.parameters(), lr=1e-3)

# 重建阶梯学习率调度器：step_size、gamma等衰减参数必须和训练保存时保持一致
# 内部步数计数会从断点恢复，不是从头重新计数
resume_sched = optim.lr_scheduler.StepLR(resume_opt, step_size=10, gamma=0.1)

# ====================== 2. 加载断点快照 ======================
# 读取本地断点文件，map_location=device 强制将文件内所有张量映射到当前运行设备
# 解决：断点保存在其他显卡/CPU，直接加载报CUDA设备不匹配的报错
ckpt = torch.load("train_checkpoint.pth", map_location=device)

# ====================== 3. 逐层回填训练状态，恢复完整训练上下文 ======================
# 从断点字典取出网络权重state_dict，回填进新建空网络，恢复模型所有参数与BN统计量
resume_model.load_state_dict(ckpt["model_state_dict"])

# 恢复优化器内部状态：Adam的一阶、二阶动量缓存、梯度累计信息
# 若跳过这一步，优化器会从零初始化，续训时loss剧烈震荡，无法延续原有训练节奏
resume_opt.load_state_dict(ckpt["optimizer_state_dict"])

# 恢复学习率调度器内部计数器（记录已衰减多少轮）
# 保证续训后学习率衰减节奏和中断前连续，不会重复衰减/跳过衰减
resume_sched.load_state_dict(ckpt["scheduler_state_dict"])

# ====================== 4. 读取训练元数据，确定续训起点 ======================
# 断点记录的是已完成的epoch，下一轮训练需要+1作为起始轮次
start_epoch = ckpt["epoch"] + 1

# 读取训练过程中记录的全局最优损失，用于后续对比更新最优模型
min_loss = ckpt["best_loss"]

# 打印恢复信息，方便日志查看断点恢复进度
print(f"恢复训练起始epoch：{start_epoch}，历史最优loss：{min_loss}")

恢复训练起始epoch：21，历史最优loss：0.12


## 4.4 TorchScript 固化计算图（无Python依赖跨端部署）
绕开pickle与Python网络类、_modules结构依赖，直接固化有序前向计算图+权重二进制数据，支持C++、移动端、边缘设备部署，无代码执行安全风险。
两种导出模式：trace（基于样例输入追踪流程）、script（完整代码编译）。

In [9]:
# trace模式导出TorchScript模型
model.eval()
sample_input = torch.randn(1, 10)
traced_script_model = torch.jit.trace(model, sample_input)
traced_script_model.save("deploy_traced.pt")
print("无Python依赖部署脚本模型导出完成")

无Python依赖部署脚本模型导出完成


In [10]:
# 加载TorchScript，无需定义任何网络类
jit_infer_model = torch.jit.load("deploy_traced.pt", map_location="cpu")
jit_infer_model.eval()
out = jit_infer_model(torch.randn(1, 10))
print("TorchScript 推理输出：", out)

TorchScript 推理输出： tensor([[-0.0633, -0.3852]], grad_fn=<AddmmBackward0>)


# 五、工程兼容适配场景
## 5.1 CPU/GPU跨设备加载 map_location
pickle说明书永久记录保存时张量绑定的设备ID，跨设备读取必须指定map_location做设备映射，否则直接报CUDA上下文错误。
### 常用映射写法
1. 全量加载至CPU：map_location="cpu"
2. 指定单卡加载：map_location="cuda:0"
3. 显卡迁移映射：map_location={"cuda:1":"cuda:0"}

In [11]:
# 1. 全部权重加载到CPU
cpu_weights = torch.load("std_weights.pth", map_location=torch.device("cpu"))
# 2. 多显卡环境映射cuda:1权重至cuda:0
migrate_weights = torch.load("std_weights.pth", map_location={"cuda:1":"cuda:0"})
print("跨设备权重加载示例执行完成")

跨设备权重加载示例执行完成


## 5.2 多卡DP/DDP module.前缀权重兼容
多卡并行训练时，DDP外层自动套一层Module包装，顶层_modules新增`module`键，所有参数名自动添加`module.`前缀，单卡直接加载会触发Missing Key报错；核心处理逻辑：保存/加载时批量去除前缀。

In [12]:
from torch.nn import DataParallel

# 模拟多卡并行模型
dp_train_model = SimpleNet()
if torch.cuda.device_count() > 1:
    dp_train_model = DataParallel(dp_train_model.cuda())

# 保存时批量剔除module.前缀，生成单卡兼容权重
raw_dp_dict = dp_train_model.state_dict()
single_compat_dict = {k.replace("module.", ""): v for k, v in raw_dp_dict.items()}
torch.save(single_compat_dict, "dp_single_compat.pth")
print("多卡权重去前缀保存完成")

多卡权重去前缀保存完成


In [13]:
# 加载带module前缀的权重至单卡网络
single_infer_net = SimpleNet()
loaded_dp = torch.load("dp_single_compat.pth", map_location="cpu")

# 通用去前缀修复有序参数名
fix_dict = {}
for k, v in loaded_dp.items():
    if k.startswith("module."):
        fix_dict[k[7:]] = v
    else:
        fix_dict[k] = v

single_infer_net.load_state_dict(fix_dict)
print("多卡权重单卡兼容加载完成")

多卡权重单卡兼容加载完成


## 5.3 迁移学习 strict=False 局部权重加载
预训练网络与下游任务网络_modules层结构不完全一致时，开启strict=False，仅匹配名称完全相同的有序权重，自动忽略新增/删除层参数，实现局部权重迁移。

In [25]:
# 1. 读取完整预训练所有权重（包含预训练模型全部层参数）
pretrain_weight = torch.load("std_weights.pth", map_location="cpu")
# 2. 搭建下游任务新网络，随机初始化一套全新权重
downstream_model = SimpleNet()
# 取出下游网络自身所有参数名、随机权重
target_dict = downstream_model.state_dict()

# 核心：只保留「预训练、下游网络两边同时存在」的参数
# 不在下游网络里的参数直接丢弃，不会载入
matched_weight = {k: v for k, v in pretrain_weight.items() if k in target_dict}
# 把匹配成功的预训练权重，覆盖进下游网络的参数字典
target_dict.update(matched_weight)

# strict=False：允许参数不完全匹配（有参数没覆盖、有多余参数都不报错）
downstream_model.load_state_dict(target_dict, strict=False)
print("迁移学习局部权重加载完成")

迁移学习局部权重加载完成


# 六、高频报错完整避坑指南
1. Missing key / Unexpected key
   根因：多卡modules带module前缀、网络_modules层定义顺序修改、预训练网络结构差异
   解决方案：批量替换module.前缀；迁移学习使用strict=False

2. CUDA设备不匹配报错
   根因：权重文件绑定原始显卡ID，当前环境无对应显卡
   解决方案：所有torch.load强制传入map_location参数

3. 完整model加载报类找不到
   根因：pickle说明书记录本地网络类文件路径，加载环境无对应源码重建_modules结构
   解决方案：生产环境统一只用state_dict存储权重，禁用save(model)

4. 加载权重后训练精度暴跌
   根因：未恢复优化器/调度器状态；推理阶段未执行model.eval()冻结BN缓冲区；张量设备不统一

5. PyTorch2.6+ UnpicklingError拦截报错
   根因：完整model携带_modules重建执行代码，触发安全拦截，默认weights_only=True
   解决方案：线上仅使用state_dict/safetensors；本地临时调试添加weights_only=False

# 七、通用工程工具函数封装（项目直接复用）

In [15]:
# 导入PyTorch核心库、网络层模块、优化器相关工具
import torch
import torch.nn as nn
import torch.optim as optim

def save_compatible_weight(model: nn.Module, save_path: str):
    """
    自动去除module前缀，保存单卡兼容state_dict权重
    适用场景：DP/DDP多卡训练模型导出权重，单卡环境可直接加载无Missing Key报错
    :param model: 训练用网络实例（支持单卡、DataParallel、DistributedDataParallel多卡模型）
    :param save_path: 权重文件保存路径，后缀一般为.pth
    """
    # 读取模型原始权重字典，多卡训练时所有参数key会自带 "module." 前缀
    raw_state = model.state_dict()
    # 字典推导式批量清洗参数名：把所有key中的 "module." 字符串删除
    # 处理后权重文件单卡网络无需额外清洗，直接load_state_dict
    clean_state = {k.replace("module.", ""): v for k, v in raw_state.items()}
    # 序列化纯权重字典写入磁盘，无pickle执行代码，生产安全
    torch.save(clean_state, save_path)
    # 控制台日志，标记文件保存完成与路径
    print(f"[保存兼容权重] {save_path}")


def load_model_weight(model: nn.Module, weight_path: str, device="cpu") -> nn.Module:
    """
    通用权重加载，自动处理设备映射，封装推理标准化流程
    :param model: 空的、结构完全匹配的网络实例（未初始化权重）
    :param weight_path: 本地权重文件路径
    :param device: 目标运行设备，默认cpu，可传入"cuda:0"等
    :return: 加载完成权重、迁移至目标设备、切换为eval推理模式的模型
    """
    # 字符串设备标识转为标准torch.device设备对象
    dev = torch.device(device)
    # 读取权重文件：map_location统一把张量映射到目标设备；weights_only=True开启PyTorch2.6安全模式，拦截恶意pickle
    w_map = torch.load(weight_path, map_location=dev, weights_only=True)
    # strict=True严格匹配参数名，参数缺失/多余直接抛出异常，快速定位网络结构不匹配问题
    model.load_state_dict(w_map, strict=True)
    # 将完整模型迁移至目标设备GPU/CPU
    model.to(dev)
    # 切换推理模式：关闭Dropout、冻结BN层running_mean/running_var统计量，保证推理稳定
    model.eval()
    # 打印加载日志，记录文件路径与目标设备
    print(f"[加载权重] {weight_path} 映射至 {device}")
    # 返回处理完毕的推理模型，支持链式调用
    return model


def save_train_checkpoint(model, optimizer, scheduler, epoch, loss, save_path):
    """
    保存完整可续训训练断点，自动兼容多卡权重
    包含网络、优化器、学习率调度器、训练元数据，中断后可无缝恢复训练
    :param model: 正在训练的网络（支持多卡DP/DDP）
    :param optimizer: 当前训练使用的优化器实例(Adam/SGD等)
    :param scheduler: 学习率调度器实例(StepLR/CosineAnnealingLR等)
    :param epoch: 当前已训练完成的轮次
    :param loss: 当前全局最优损失值，用于保存最优模型判断
    :param save_path: 断点文件存储路径
    """
    # 清洗多卡模型自带的module.前缀，保证单卡恢复训练无参数名匹配错误
    clean_w = {k.replace("module.", ""): v for k, v in model.state_dict().items()}
    # 打包完整训练快照字典，所有续训必须的状态全部存入
    ckpt = {
        "epoch": epoch,                  # 已完成训练轮次，恢复时+1作为起始轮
        "best_loss": loss,               # 历史最优损失，用于早停、最优模型筛选
        "model_state": clean_w,          # 清洗后的网络所有权重与BN缓冲区
        "opt_state": optimizer.state_dict(), # 优化器内部动量、梯度统计状态，保证续训不震荡
        "sched_state": scheduler.state_dict() # 学习率调度器计数，保证衰减节奏连续
    }
    # 序列化完整断点字典保存到本地
    torch.save(ckpt, save_path)
    # 断点保存日志输出
    print(f"[保存训练断点] {save_path}")


# 工具调用示例入口：仅当前脚本直接运行时执行，被其他文件import导入不会执行
if __name__ == "__main__":
    # 实例化基础测试网络 SimpleNet
    net = SimpleNet()
    # 创建Adam优化器，绑定网络全部可训练参数
    opt = optim.Adam(net.parameters())
    # 余弦退火学习率调度器，T_max=10代表10个epoch完成一轮余弦周期衰减
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10)

    # 调用封装工具：保存兼容多卡的纯推理权重文件
    save_compatible_weight(net, "tool_std_weight.pth")
    # 调用封装工具：保存完整训练断点快照
    save_train_checkpoint(net, opt, epoch=5, loss=0.3, save_path="tool_ckpt.pth")

    # 新建空推理网络，结构与训练网络保持一致
    infer_net = SimpleNet()
    # 调用通用加载工具，自动完成设备映射、权重回填、切换eval推理模式
    infer_net = load_model_weight(infer_net, "tool_std_weight.pth")

[保存兼容权重] tool_std_weight.pth


TypeError: save_train_checkpoint() missing 1 required positional argument: 'scheduler'

# 八、拓展：safetensors 无pickle安全权重格式
基于前文pickle代码执行漏洞痛点设计，完全抛弃pickle序列化逻辑，仅存储「参数名-二进制张量」静态映射，不读取模型_modules结构与重建代码，无任何可执行指令，彻底规避UnpicklingError安全拦截，是大模型、开源分发工业标准，原生兼容PyTorch。

In [ ]:
# 安装依赖：pip install safetensors
try:
    from safetensors.torch import save_file, load_file
    # 保存安全无pickle权重
    save_file(model.state_dict(), "safe_weights.safetensors")
    # 加载安全权重
    safe_weight = load_file("safe_weights.safetensors")
    model.load_state_dict(safe_weight)
    print("safetensors 安全权重读写完成，无pickle执行风险")
except ImportError:
    print("未安装safetensors，执行 pip install safetensors 启用安全格式")

# 九、全文核心应用主线：对外交付模型三大完整方案选型对比
## 前置总览（基于前文全部底层原理做落地分发决策）
1. 方案A：单文件state_dict权重 weights.pth（纯原生，无配套结构文件）
2. 方案B：原生双文件 weights.pth + model_arch.py（权重+配套网络源码，团队内部）
3. 方案C：Hugging Face一体化包 config.json + model.safetensors（开源商用交付行业标准）

## 方案A：单文件weights.pth（state_dict原生权重）
### 底层依托前文逻辑
仅存储静态参数映射说明书，不序列化model._modules网络结构，对应4.1标准state_dict存储场景。
### 保存代码
```python
torch.save(model.state_dict(), "weights.pth")
```
### 加载硬性约束
使用者必须完整复刻和训练时完全一致的nn.Module网络类，保证_modules层顺序、参数维度不能改动，否则参数名不匹配报错。
### 优缺点
✅ 无第三方依赖、文件体积最小、weights_only安全；
❌ 单独文件无网络结构信息，分发极易丢失网络源码导致失效；
适用：仅个人本地调试，绝不对外交付。

## 方案B：双文件 weights.pth + model_arch.py
### 底层依托前文逻辑
在方案A安全权重基础上，配套独立网络定义py文件（内部定义完整nn.Module，自带_modules层结构），把「有序权重映射」和「网络结构定义」拆分交付，使用者直接导入架构，无需手写网络。
### 文件结构
release/
├─ weights.pth
└─ model_arch.py # 完整SimpleNet类定义，自带网络_modules结构
### 用户加载代码
```python
from model_arch import SimpleNet
net = SimpleNet()
sd = torch.load("weights.pth", map_location="cpu", weights_only=True)
net.load_state_dict(sd)
```
### 优缺点
✅ 纯原生无第三方库、支持二次微调、权重安全；
❌ 两份文件缺一不可，碎片化转发容易丢失架构文件；
适用：团队内部小范围分享、允许对方二次微调、禁止安装transformers。

## 方案C：HF一体化 config.json + model.safetensors
### 底层依托前文逻辑
1. 用config.json标准化存储网络超参、层结构信息，替代手写nn.Module与_modules源码；
2. 使用safetensors无pickle权重格式，规避安全拦截；
3. AutoModel依靠配置文件自动重建等价_modules网络结构，用户零手写网络代码；
### 标准目录
hf_model/
├─ config.json # 标准化网络结构配置，替代model_arch.py
├─ model.safetensors # 安全权重
└─ README.md
### 保存代码（transformers封装PreTrainedModel）
```python
from transformers import PreTrainedModel, PretrainedConfig
class SimpleNetConfig(PretrainedConfig):
    model_type = "simplenet"
    def __init__(self, input_dim=10, output_dim=2, **kwargs):
        super().__init__(**kwargs)
        self.input_dim = input_dim
        self.output_dim = output_dim
class SimpleNet(PreTrainedModel):
    config_class = SimpleNetConfig
    def __init__(self, config):
        super().__init__(config)
        self.linear = nn.Linear(config.input_dim, config.output_dim)
    def forward(self, x):
        return self.linear(x)
config = SimpleNetConfig()
model = SimpleNet(config)
model.save_pretrained("./hf_model", safe_serialization=True)
```
### 用户一行加载
```python
from transformers import AutoModel
model = AutoModel.from_pretrained("./hf_model")
```
### 优缺点
✅ 配置文件自带完整网络结构、safetensors无pickle风险、支持一键上传HF Hub、开源商用行业标准；
❌ 依赖transformers第三方库；
适用：论文开源、客户交付、大模型长期维护。

## 三种交付方案完整对比总表
| 对比维度 | 方案A 单文件weights.pth | 方案B 双文件weights+model_arch.py | 方案C HF一体化包 |
| ---- | ---- | ---- | ---- |
| 是否内置网络_modules结构 | ❌ 无，需手写复刻 | ❌ 单独py文件配套 | ✅ config.json内置 |
| 用户是否手写完整网络类 | ✅ 必须完整复刻 | ❌ 直接导入文件 | ❌ 零网络代码 |
| 支持二次微调 | ✅ 支持 | ✅ 支持 | ✅ 支持 |
| 第三方库依赖 | 无 | 无 | 需要transformers |
| 文件丢失分发风险 | 极高 | 中等 | 极低（完整仓库打包） |
| pickle安全风险 | ✅ weights_only规避 | ✅ weights_only规避 | ✅ safetensors彻底无pickle |
| 开源商用交付适配 | ❌ 不推荐 | ⚠️ 仅内部小团队 | ✅ 行业标准首选 |